# 📘 부록: RunPod GPU에서 실습하기

이 단원부터 과정 끝까지의 실습(딥러닝 학습, LLM 파인튜닝, 모델 서빙)은 GPU에서 돌립니다. 개인 PC에는 GPU가 없거나 메모리가 부족한 경우가 많아, 필요한 만큼만 GPU를 빌려 쓰는 클라우드 서비스 RunPod를 씁니다. 이 부록은 RunPod에서 GPU 서버를 만들고, 수업 때 쓰던 VS Code로 그 서버에 접속해 교안을 실행하고, 끝나면 비용이 더 나가지 않게 정리하는 방법을 정리합니다.

**이 부록의 흐름**
- RunPod의 용어(파드, 템플릿, 스토리지)와 요금이 나가는 방식, GPU를 고르는 기준을 봅니다.
- 처음 한 번만 SSH 키를 만들어 RunPod에 등록하고 VS Code에 Remote - SSH 확장을 설치합니다.
- 파드를 만들고 VS Code로 접속합니다.
- 터미널에서 `git clone` 으로 실습 자료를 받고 `uv sync` 로 파이썬 환경을 만듭니다.
- GPU가 잡혔는지 확인하고 교안을 실행한 뒤, 결과를 내려받고 파드를 정지·종료합니다.

## 1. RunPod 한눈에 보기

### 1-1. 파드와 템플릿

- **파드(Pod):** RunPod에서 빌리는 GPU 서버 한 대입니다. GPU 종류와 개수, 디스크 크기를 정해 만들고, 켜져 있는 시간만큼 요금이 나갑니다(밀리초 단위로 정산).
- **템플릿(Template):** 파드에 미리 설치해 둘 프로그램 묶음입니다. 이 과정은 PyTorch·CUDA와 SSH 접속, Jupyter Lab, git·uv가 갖춰진 공식 PyTorch 템플릿을 씁니다.

### 1-2. 저장 공간: 무엇이 남고 무엇이 지워지나

파드 만들기 화면의 **Storage** 칸에서 저장 공간 두 가지를 정합니다. 파드를 정지(Stop)하거나 종료(Terminate)할 때 남는 것이 다릅니다.

| 저장 공간 | 무엇인가 | 정지하면 | 종료하면 | 요금 |
|---|---|---|---|---|
| 컨테이너 디스크(Container disk) | 파드에 딸린 기본 디스크(기본 50GB). 올린 파일과 설치한 패키지가 모두 여기에 저장됩니다 | 지워짐 | 지워짐 | 파드가 켜져 있는 동안만 |
| 네트워크 볼륨(Persistent storage) | 계정에 따로 만들어 두는 디스크. 파드에 연결하면 `/workspace`에 보입니다 | 남음 | 남음 | 지우기 전까지 매달 |

- **이 단원은 네트워크 볼륨 없이 컨테이너 디스크만 씁니다.** 데이터가 작아 다시 올리기 쉽기 때문입니다. 대신 정지하는 순간 파일이 모두 지워지므로 실습 결과는 끝내기 전에 내려받습니다.
- **모델을 내려받는 파인튜닝 단원부터는 네트워크 볼륨을 씁니다.** 내려받은 모델이 볼륨에 남아 다음 수업에 다시 받지 않아도 됩니다. 볼륨은 만든 지역(region)에 묶이므로 그 지역에 있는 GPU만 고를 수 있습니다.
- **크레딧이 0이 되면 파드가 멈추고, 네트워크 볼륨이 없는 파드는 데이터를 되살릴 수 없습니다.**

### 1-3. 요금이 나가는 방식

- **실행 중:** GPU 요금과 컨테이너 디스크 요금이 나갑니다. 브라우저를 닫아도 파드가 켜져 있으면 계속 나갑니다.
- **정지 중:** GPU 요금은 멈춥니다. 볼륨 없이 만든 파드는 정지 중 요금이 없습니다.
- **종료 뒤:** 파드 요금은 더 나가지 않습니다. 네트워크 볼륨은 파드와 따로 남아 지울 때까지 요금이 나갑니다.
- **배포 조건:** 크레딧이 있어야 파드를 만들 수 있습니다. GPU 가격은 바뀌므로 파드를 만들 때 화면에 보이는 시간당 가격을 확인합니다.

## 2. GPU 고르기

GPU를 고를 때 가장 먼저 보는 것은 **GPU 메모리(VRAM)** 입니다. 모델과 학습 데이터가 GPU 메모리에 다 들어가야 학습이 돌고, 메모리가 모자라면 `CUDA out of memory` 오류로 멈춥니다. 메모리가 충분하면 그다음은 세대(같은 일을 얼마나 빨리 하는가)와 시간당 가격을 봅니다.

### 2-1. 메모리 어림 계산

- **모델 불러오기:** 16비트로 불러온 모델은 파라미터 10억 개마다 약 2GB를 씁니다. 8B 모델이면 약 16GB입니다.
- **학습:** 기울기와 옵티마이저가 기억하는 값(AdamW는 파라미터마다 두 개)까지 더해져 불러오기만 할 때보다 몇 배 더 씁니다. 그래서 큰 모델은 LoRA·4비트 QLoRA처럼 학습할 값을 줄이는 방법을 씁니다.
- **배치 크기:** 배치를 키우면 한 번에 GPU에 올리는 데이터가 늘어 메모리를 더 씁니다. 메모리가 모자라면 배치부터 줄입니다.

### 2-2. RunPod의 GPU별 특징

| 메모리 | GPU | 시간당 가격(대략) | 학습 관점의 특징 |
|---|---|---|---|
| 16GB | RTX 2000 Ada·RTX A4000 | 0.25달러 | 가장 저렴합니다. 느리지만 작은 신경망·PyTorch 연습에는 충분합니다 |
| 20GB | RTX A4500·RTX 4000 Ada | 0.25~0.3달러 | 16GB와 값이 비슷하고 메모리가 조금 더 큽니다 |
| 24GB | RTX A5000 | 0.27달러 | 24GB 중 가장 저렴합니다. 7B 모델 LoRA·QLoRA 연습의 기본 선택입니다 |
| 24GB | RTX 3090·RTX 4090 | 0.5~0.9달러 | 게이밍용 고성능 카드입니다. 같은 24GB에서 학습이 가장 빠르고, 4090이 3090보다 빠릅니다 |
| 24GB | L4 | 0.6달러 | 전력이 낮은 추론(서빙)용 카드입니다. 메모리 대역폭이 좁아 학습은 A5000보다 느립니다 |
| 32GB | RTX 5090 | 1.2달러 | 최신 게이밍 카드입니다. 24GB로 조금 모자랄 때 고릅니다 |
| 48GB | A40·RTX A6000 | 0.6달러 | 48GB 중 가장 저렴합니다. 8B 모델을 16비트로 띄우는 서빙 실습에 맞습니다 |
| 48GB | L40S·RTX 6000 Ada | 1~1.1달러 | A40보다 한 세대 새 카드라 학습·추론이 더 빠릅니다 |
| 80GB | A100 | 1.8달러 | 데이터센터용입니다. 빠른 HBM 메모리를 써서 큰 모델 학습이 빠릅니다 |
| 80GB 이상 | H100·H200·B200 | 3~8달러 | 최상위 데이터센터용입니다. 대형 모델 사전학습·다중 GPU 학습에 쓰며 이 과정에는 필요하지 않습니다 |

- **세대:** Ampere(RTX A·3090·A40·A100)에서 Ada(RTX Ada·4090·L4·L40S)·Hopper(H100·H200)를 거쳐 Blackwell(5090·B200)로 이어집니다. Ada에는 그래픽 카드(RTX 4090·RTX Ada)와 데이터센터 카드(L4·L40S)가 함께 있고, Hopper는 데이터센터 전용입니다. 같은 계열이면 새 세대가 학습이 빠르지만, 계열이 다르면 옛 세대가 빠를 수도 있습니다(A100이 L4보다 훨씬 빠릅니다).
- **정밀도:** 이 과정이 쓰는 16비트(bf16) 학습은 Ampere부터 모두 됩니다. Ada·Hopper부터 FP8, Blackwell부터 FP4 같은 더 낮은 정밀도 계산을 하드웨어가 직접 지원합니다.
- **이름의 뜻:** 이름에 SXM이 붙은 카드는 GPU끼리 빠르게 연결돼 여러 장으로 학습할 때 유리하고, PCIe는 한 장으로 쓸 때 충분합니다. MIG는 큰 GPU 한 장을 여러 조각으로 나눈 것이라 이름에 적힌 만큼의 메모리와 그에 맞는 연산 능력만 씁니다.
- **AMD GPU:** MI300X 같은 AMD GPU는 CUDA가 아니라 ROCm을 씁니다. 이 과정의 코드와 PyTorch 템플릿은 NVIDIA(CUDA) 기준이므로 NVIDIA GPU를 고릅니다.
- **가격과 자리:** 위 가격은 파드 만들기 화면에 보이는 대략적인 값이고 수시로 바뀝니다. 싼 GPU는 빈 자리가 없을 때가 많습니다(카드에 **Out of capacity** 표시). GPU 목록의 **Available** 탭에는 지금 빌릴 수 있는 GPU만 보입니다.

### 2-3. 이 과정의 단계별 권장 GPU

| 단계 | 실습 | 권장 GPU 메모리 |
|---|---|---|
| 딥러닝 학습 원리·PyTorch·트랜스포머 | 작은 텐서와 신경망 | 16GB(RTX 2000 Ada·RTX A4000) |
| 파인튜닝(SFT·LoRA·QLoRA·DPO)·임베딩 파인튜닝 | 1.7B~8B 모델 학습(8B는 4비트 QLoRA) | 24GB(RTX A5000·RTX 3090·RTX 4090) |
| 서빙(vLLM)·벤치마크·배포 | 8B 모델을 16비트로 띄워 요청 처리 | 48GB(A40·RTX A6000·L40S) |

이 단원은 데이터가 작아 가장 저렴한 16GB GPU(RTX 2000 Ada·RTX A4000)면 충분합니다. 고르려는 GPU가 목록에 없으면 지금 빈 자리가 없다는 뜻이므로 같은 메모리의 다른 GPU를 고릅니다.

## 3. 처음 한 번만: SSH 키와 VS Code 준비

**SSH(Secure Shell)** 는 내 PC에서 원격 서버의 터미널을 암호화된 연결로 여는 방법입니다. VS Code의 **Remote - SSH** 확장은 이 연결 위에서 원격 서버의 폴더를 내 PC 폴더처럼 열어 줍니다. 그래서 수업 때 쓰던 VS Code 화면 그대로 GPU 서버의 노트북을 편집하고 실행합니다.

SSH 접속에는 열쇠 한 쌍이 필요합니다.

- **개인키(`id_ed25519`):** 내 PC에만 둡니다. 남에게 보내거나 어디에 붙여 넣지 않습니다.
- **공개키(`id_ed25519.pub`):** RunPod 계정에 등록합니다. RunPod는 파드를 만들 때 이 공개키를 파드에 넣어 두고, 같은 쌍의 개인키를 가진 PC만 들여보냅니다.

이 절의 세 가지(키 만들기, 공개키 등록, 확장 설치)는 처음 한 번만 합니다. 파드를 새로 만들어도 다시 하지 않습니다.

### 3-1. SSH 키 만들기

VS Code 위쪽 메뉴 **Terminal → New Terminal** 로 내 PC 터미널을 엽니다(macOS는 zsh, Windows는 PowerShell). 아래 명령은 macOS와 Windows에서 똑같이 씁니다.

**OpenSSH 명령(내 PC 터미널)**

```bash
# 키 한 쌍을 만듭니다. -C 뒤의 runpod 는 키에 붙이는 이름표입니다
ssh-keygen -t ed25519 -C runpod
```

- 저장 위치와 암호를 묻는 질문 세 개가 나오면 모두 Enter를 누릅니다(기본 위치, 암호 없음).
- `Overwrite (y/n)?` 가 나오면 이미 키가 있다는 뜻입니다. `n` 을 누르고 있는 키를 그대로 씁니다.

**셸 명령(내 PC 터미널)**

```bash
# 공개키 내용을 화면에 띄웁니다. ssh-ed25519 로 시작하는 한 줄 전체를 복사합니다
cat ~/.ssh/id_ed25519.pub
```

<img src="images/vscode_01_keygen.png" width="1050" alt="VS Code 터미널에서 ssh-keygen 을 실행한 화면. Enter로 넘긴 질문 세 줄과 cat 으로 띄운 공개키 한 줄에 빨간 사각형이 그려져 있다.">

### 3-2. RunPod에 공개키 등록하기

공개키는 **파드를 만들기 전에** 등록합니다. 이미 만든 파드에는 나중에 등록한 키가 들어가지 않습니다.

1. RunPod 콘솔 왼쪽 메뉴 **Credentials** 를 누르고 **SSH Public Keys** 탭에서 **Add SSH key** 를 누릅니다.
2. **Title** 에 알아보기 쉬운 이름(`my-laptop` 등)을, **Key** 에 3-1에서 복사한 공개키 한 줄을 붙여 넣고 **Add SSH key** 를 누릅니다.

<img src="images/runpod_10_ssh_keys.png" width="960" alt="RunPod 콘솔 Credentials 화면. 왼쪽 메뉴 Credentials, SSH Public Keys 탭, Add SSH key 버튼에 빨간 사각형이 그려져 있다.">

<img src="images/runpod_11_add_key.png" width="560" alt="Add SSH key 창. Key 칸에 붙여 넣은 공개키와 Add SSH key 버튼에 빨간 사각형이 그려져 있다.">

### 3-3. VS Code에 Remote - SSH 확장 설치

VS Code 왼쪽 **확장(Extensions)** 아이콘을 누르고 `Remote - SSH` 를 검색해 Microsoft가 만든 확장을 **Install** 합니다. 함께 깔리는 Remote Explorer도 그대로 둡니다.

<img src="images/vscode_02_remote_ssh.png" width="720" alt="VS Code 확장 검색 화면. 검색 칸의 Remote - SSH 와 Microsoft Remote - SSH 확장의 Install 버튼에 빨간 사각형이 그려져 있다.">

## 4. 파드 만들기

### 4-1. 크레딧 충전

RunPod는 미리 충전한 크레딧에서 요금을 빼 가는 방식입니다. 콘솔 왼쪽 메뉴의 **Billing** 에서 크레딧을 충전합니다. 잔액이 줄어들 때 메일로 알려 주는 설정도 같은 화면에 있습니다.

### 4-2. 파드 만들기 화면 열기

콘솔 왼쪽 메뉴의 **Pods** 를 누르면 **Deploy a Pod** 화면이 열립니다(이미 파드가 있으면 파드 목록 위의 **Deploy** 버튼을 누릅니다). 한 화면에서 위에서부터 템플릿, 이름, 지역, GPU, 저장 공간을 정하고 오른쪽 요약 칸에서 배포합니다.

<img src="images/runpod_01_pods.png" width="1050" alt="RunPod 콘솔의 Deploy a Pod 화면. 왼쪽 메뉴의 Pods와 화면 제목 Deploy a Pod에 빨간 사각형이 그려져 있고, 아래에 PyTorch 템플릿과 Pod name 칸이 보인다.">

### 4-3. 템플릿과 이름, SSH 설정

- **Workload:** 공식 **PyTorch** 템플릿이 기본으로 골라져 있습니다. 다른 템플릿이 보이면 검색 칸에서 PyTorch를 찾아 고릅니다.
- **Pod name:** `day65-practice` 처럼 알아보기 쉽게 적습니다.
- **Set overrides:** 누르면 **Edit template overrides** 창이 열립니다. 아래 세 가지가 켜져 있는지 확인만 하고 오른쪽 위 X로 닫습니다.
  - **SSH terminal access:** VS Code 접속에 필요합니다.
  - **Exposed TCP ports의 22:** SSH가 쓰는 포트입니다. 이 포트가 열려 있어야 5절의 접속 명령이 나옵니다.
  - **Start Jupyter notebook:** VS Code 접속이 안 될 때 쓰는 비상 경로(9절)입니다.

<img src="images/runpod_02_template.png" width="560" alt="Edit template overrides 창. Start Jupyter notebook 과 SSH terminal access 체크 상자, Exposed TCP ports 의 22 에 빨간 사각형이 그려져 있다.">

### 4-4. GPU 고르기

- **Region** 은 **Any region** 으로 둡니다. 자리가 있는 모든 지역의 GPU가 보입니다.
- **Compute** 의 **GPU** 탭에서 **Search GPUs** 칸에 `RTX 2000` 이나 `A4000` 을 입력하고 **Available**(지금 빌릴 수 있는 GPU) 탭을 누른 뒤, 2절 기준에 맞는 카드를 누릅니다.
- 그 아래 **GPU count** 는 1, **Instance pricing** 은 쓴 만큼 내는 **On-Demand** 로 둡니다.

<img src="images/runpod_03_gpu.png" width="875" alt="Compute 칸의 GPU 목록. 검색 칸의 RTX 2000 Ada, Available 탭, 고른 GPU 카드(RTX 2000 Ada, 16GB, 시간당 0.24달러)에 빨간 사각형이 그려져 있다.">

### 4-5. 저장 공간과 배포

- **Container disk:** 기본값 50GB로 둡니다. 6절에서 만드는 파이썬 환경(약 7GB)도 여기에 들어갑니다.
- **Persistent storage:** 이 단원은 비워 둡니다. 요약 칸에 `Nothing mounted at the template's path` 가 떠도 그대로 배포합니다.
- 오른쪽 요약 칸의 **Total / hour**(GPU와 디스크를 합한 시간당 요금)를 확인하고 **Deploy Pod** 를 누릅니다. 버튼이 **Deploy when available** 로 바뀌어 있으면 고른 GPU에 빈 자리가 없다는 뜻이므로 누르지 말고 다른 GPU를 고릅니다.

<img src="images/runpod_04_summary.png" width="320" alt="배포 직전 오른쪽 요약 칸. Total / hour 0.25달러와 Deploy Pod 버튼에 빨간 사각형이 그려져 있다.">

## 5. VS Code로 파드에 접속하기

### 5-1. 접속 명령 복사

배포하면 Pods 화면 오른쪽에 파드 상세 칸이 열립니다. 파드가 시작되는 데 1~2분 걸립니다. **Connect** 탭의 **SSH over exposed TCP** 아래 명령(`ssh root@주소 -p 포트 -i ~/.ssh/id_ed25519`)을 오른쪽 복사 버튼으로 복사합니다.

- 위쪽 **SSH** 칸의 명령(`...@ssh.runpod.io`)은 파일 전송을 지원하지 않아 VS Code 접속에 쓸 수 없습니다. 반드시 **SSH over exposed TCP** 의 명령을 씁니다.
- **SSH over exposed TCP** 칸이 없으면 4-3의 SSH 설정이 꺼진 채 만든 파드입니다. 종료하고 설정을 켠 채 새로 만듭니다.

<img src="images/runpod_05_connect.png" width="598" alt="실행 중인 파드의 상세 칸. Connect 탭과 SSH over exposed TCP 제목부터 접속 명령까지에 빨간 사각형이 그려져 있다.">

### 5-2. VS Code에 호스트 추가

1. VS Code에서 명령 팔레트(macOS `Cmd+Shift+P`, Windows `Ctrl+Shift+P`)를 열고 `Remote-SSH: Connect to Host` 를 입력해 **Remote-SSH: Connect to Host...** 를 고릅니다.
2. **+ Add New SSH Host...** 를 고릅니다.
3. 5-1에서 복사한 명령을 붙여 넣고 Enter를 누릅니다.
4. 저장할 설정 파일은 첫 번째(내 사용자 폴더의 `.ssh/config`)를 고릅니다. 오른쪽 아래에 `Host added!` 가 뜨면 끝입니다.

<img src="images/vscode_03_connect_host.png" width="640" alt="명령 팔레트에 Remote-SSH: Connect to Host 를 입력한 화면. 입력 칸과 Remote-SSH: Connect to Host... 항목에 빨간 사각형이 그려져 있다.">

<img src="images/vscode_04_add_host.png" width="640" alt="Add New SSH Host... 항목에 빨간 사각형이 그려진 화면.">

<img src="images/vscode_05_ssh_command.png" width="640" alt="Enter SSH Connection Command 입력 칸에 ssh root@주소 -p 포트 -i ~/.ssh/id_ed25519 명령을 붙여 넣은 화면.">

<img src="images/vscode_06_config_file.png" width="640" alt="Select SSH configuration file to update 목록. 첫 번째 사용자 폴더의 .ssh/config 에 빨간 사각형이 그려져 있다.">

### 5-3. 접속하고 작업 폴더 열기

1. 다시 **Remote-SSH: Connect to Host...** 를 열면 목록에 파드 주소가 생겨 있습니다. 주소를 고르면 새 창이 열리며 접속합니다.
2. 처음 접속하면 위쪽 입력 칸에 `The authenticity of host ... can't be established. Are you sure you want to continue connecting (yes/no/[fingerprint])?` 가 뜹니다. 처음 보는 서버라 확인하는 질문이므로 `yes` 를 입력하고 Enter를 누릅니다(입력한 글자는 점으로 가려 보입니다).
3. 왼쪽 아래에 `SSH: 주소` 가 보이면 접속된 것입니다. 첫 접속은 파드에 VS Code 서버를 설치하느라 1분쯤 걸립니다.
4. **File → Open Folder...** 에서 `/workspace/` 를 입력하고 **OK** 를 누릅니다. 폴더를 믿을지 묻는 창이 뜨면 **Yes, I trust the authors** 를 누릅니다(No를 누르면 확장이 일부만 동작합니다).

<img src="images/vscode_07_pick_host.png" width="640" alt="Connect to Host 목록. 새로 추가된 파드 주소에 빨간 사각형이 그려져 있다.">

<img src="images/vscode_08_open_folder.png" width="640" alt="원격 창의 Open Folder 입력 칸. /workspace/ 와 OK 버튼에 빨간 사각형이 그려져 있다.">

### 5-4. 파드에 Python·Jupyter 확장 설치

확장은 내 PC와 파드에 따로 설치됩니다. 접속한 창의 확장 화면에서 `Python` 을 검색하면, 내 PC에만 있는 확장이 회색으로 보이고 **Install in SSH: 주소** 버튼이 붙어 있습니다. **Python** 을 **Install in SSH** 로 설치하고, 이어서 `Jupyter` 로 다시 검색해 같은 방법으로 설치합니다.

<img src="images/vscode_09_remote_ext.png" width="560" alt="접속한 창의 확장 검색 화면. 검색 칸의 Python 과 Python 확장의 Install in SSH 버튼에 빨간 사각형이 그려져 있다.">

## 6. 실습 자료 받기와 파이썬 환경 만들기

접속한 창에서 **Terminal → New Terminal** 을 열면 파드의 리눅스 터미널(bash)이 열립니다. 이 절의 명령은 모두 이 터미널에서 실행합니다. 명령은 하는 일에 따라 네 종류입니다.

| 종류 | 하는 일 | 이 절의 명령 |
|---|---|---|
| 리눅스 셸 명령 | 폴더 이동, 목록 보기, 파일 받기 | `cd`, `ls`, `curl` |
| git 명령 | GitHub 저장소 받기 | `git clone` |
| uv 명령 | 파이썬 환경 만들기, 그 환경으로 실행 | `uv sync`, `uv run` |
| apt 명령 | 리눅스 프로그램 설치(관리자 권한) | `apt-get update`, `apt-get install` |

### 6-1. 실습 자료 받기

**리눅스 셸 명령(파드 터미널)**

```bash
# 작업 폴더로 이동합니다
cd /workspace
```

**git 명령(파드 터미널)**

```bash
# 과정 실습 저장소를 받습니다. enkoa-practice-deep-learning 폴더가 생깁니다
git clone https://github.com/paircodingofficial-cloud/enkoa-practice-deep-learning.git
```

**리눅스 셸 명령(파드 터미널)**

```bash
# 받은 폴더에 pyproject.toml, uv.lock(설치할 패키지와 버전 목록)과 단원 폴더가 있는지 봅니다
ls enkoa-practice-deep-learning
```

<img src="images/vscode_10_clone.png" width="1050" alt="파드 터미널에서 git clone 을 실행한 화면. git clone 명령과 ls 로 본 저장소 파일 목록, 왼쪽 탐색기의 enkoa-practice-deep-learning 폴더에 빨간 사각형이 그려져 있다.">

### 6-2. 파이썬 환경 만들기

**리눅스 셸 명령(파드 터미널)**

```bash
# 받은 저장소 폴더로 들어갑니다
cd enkoa-practice-deep-learning
```

**uv 명령(파드 터미널)**

```bash
# pyproject.toml·uv.lock 에 적힌 패키지를 .venv 폴더에 그대로 설치합니다(약 1분)
uv sync
```

```bash
# 방금 만든 환경의 PyTorch 버전과 GPU 사용 여부를 찍어 봅니다
uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
```

- `uv sync` 는 PyTorch를 CUDA 12.8 빌드(`torch==2.11.0+cu128`)로 설치합니다. 마지막 줄이 `2.11.0+cu128 True` 이면 GPU를 쓸 준비가 된 것입니다.
- 왼쪽 탐색기에 `.venv` 폴더가 생깁니다. 7절에서 노트북 커널로 이 환경을 고릅니다.

<img src="images/vscode_11_uv_sync.png" width="1050" alt="uv sync 가 끝난 터미널. 설치된 torch==2.11.0+cu128 줄, uv run 으로 찍은 2.11.0+cu128 True, 탐색기의 .venv 폴더에 빨간 사각형이 그려져 있다.">

### 6-3. 그래프 한글 글꼴 설치

**apt 명령(파드 터미널)**

```bash
# 설치할 수 있는 프로그램 목록을 새로 받습니다
apt-get update
```

```bash
# 그래프의 한글이 네모로 나오지 않게 나눔고딕 글꼴을 설치합니다
apt-get install -y fonts-nanum
```

### 6-4. 저장소 폴더 열기

**File → Open Folder...** 에서 `/workspace/enkoa-practice-deep-learning/` 을 열면 창이 다시 열립니다. 저장소 폴더를 열어야 VS Code가 그 안의 `.venv` 를 찾아 커널 목록에 보여 줍니다(`/workspace` 를 연 채로는 보이지 않습니다).

<img src="images/vscode_12_open_repo.png" width="640" alt="Open Folder 입력 칸에 /workspace/enkoa-practice-deep-learning/ 을 넣은 화면. 입력 칸과 OK 버튼에 빨간 사각형이 그려져 있다.">

## 7. GPU 확인하고 교안 실행하기

### 7-1. 새 노트북을 만들고 커널 고르기

1. 명령 팔레트에서 **Create: New Jupyter Notebook** 으로 새 노트북을 만들고 `내작업/GPU_확인.ipynb` 로 저장합니다.
2. 오른쪽 위 **Select Kernel** 을 누르고 **Python Environments...** 를 고릅니다.
3. 목록에서 **.venv (Python 3.12.3)** 를 고릅니다(`Recommended` 표시).

- 다른 파이썬(3.10, 3.11, 3.12.3 `/usr/bin/python3` 등)은 파드에 원래 깔린 파이썬이라 이 저장소의 패키지가 없습니다. 반드시 `.venv` 를 고릅니다.
- 한 번 고르면 다음 노트북부터는 **Select Kernel** 을 누를 때 `.venv` 가 바로 위에 나옵니다.

<img src="images/vscode_13_select_kernel.png" width="1050" alt="노트북 오른쪽 위 Select Kernel 버튼과 커널 종류 목록의 Python Environments... 에 빨간 사각형이 그려져 있다.">

<img src="images/vscode_14_venv_kernel.png" width="640" alt="Select a Python Environment 목록. Recommended 표시가 붙은 .venv (Python 3.12.3) 에 빨간 사각형이 그려져 있다.">

### 7-2. GPU가 잡혔는지 확인

7-1에서 만든 노트북에 아래 두 코드를 각각 셀에 붙여 실행합니다. `nvidia-smi` 는 GPU 이름과 메모리를 보여 주는 명령이고, 셀 맨 앞의 `!` 는 셀 안에서 터미널 명령을 실행하라는 표시입니다.

```python
# GPU 이름과 메모리, 드라이버가 지원하는 CUDA 버전을 확인합니다
!nvidia-smi
```

```python
import torch

# PyTorch가 GPU를 쓸 수 있는지, 어떤 GPU인지 확인합니다
print("PyTorch 버전:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print("GPU 이름:", torch.cuda.get_device_name(0), "| GPU 메모리(GB):", round(total_gb, 1))
```

- **PyTorch 버전이 `2.11.0+cu128` 이어야 `.venv` 입니다:** `2.8` 로 시작하면 템플릿 파이썬(`/usr/bin/python3`)을 고른 것이니 7-1대로 커널을 바꿉니다.
- **CUDA 버전이 달라 보여도 됩니다:** `nvidia-smi` 의 CUDA Version은 드라이버가 지원하는 가장 높은 버전입니다. PyTorch 버전의 `cu128`(CUDA 12.8용)보다 높으면 그대로 씁니다.
- **메모리가 조금 작게 보여도 됩니다:** 48GB 카드가 47.4로 보이는 것은 1GB를 1024MB로 나눠 계산했기 때문입니다.

<img src="images/vscode_15_gpu_check.png" width="1050" alt="VS Code에서 nvidia-smi 와 torch.cuda 확인 셀을 실행한 화면. 커널 .venv (Python 3.12.3), GPU 이름 NVIDIA RTX A6000, 메모리 49140MiB, 출력 GPU 사용 가능: True 에 빨간 사각형이 그려져 있다.">

### 7-3. 교안 실행

탐색기에서 `day65_딥러닝_학습원리/교안_01_손실함수와_경사하강.ipynb` 를 열고 커널을 `.venv` 로 고른 뒤 위에서부터 실행합니다. 준비 셀이 `계산 장치: cuda | (GPU 이름)` 을 출력하면 GPU에서 계산하고 있는 것입니다. `cpu` 가 나오면 GPU 없이 만든 파드입니다. 이 부록의 VS Code 화면은 RTX A6000 파드에서 찍어 GPU 이름이 고른 GPU와 다르게 보일 수 있습니다.

<img src="images/vscode_16_lesson_cuda.png" width="1050" alt="VS Code에서 교안 01 준비 셀과 데이터 읽기 셀을 실행한 화면. 커널 .venv (Python 3.12.3), 출력 계산 장치: cuda | NVIDIA RTX A6000, 행 수: 100 에 빨간 사각형이 그려져 있다.">

## 8. 끝내기: 결과 내려받고 정지·종료하기

실습이 끝나면 반드시 파드를 정지하거나 종료합니다. VS Code 창이나 브라우저 탭을 닫는 것만으로는 파드가 꺼지지 않아 요금이 계속 나갑니다.

| 버튼 | 하는 일 | 언제 쓰나 |
|---|---|---|
| **Stop**(정지) | GPU를 반납하고 파드 설정은 남깁니다. 볼륨 없이 만든 파드는 컨테이너 디스크가 지워져 파일이 모두 사라집니다 | 같은 설정으로 다시 켤 때 |
| **Terminate**(종료) | 파드를 지웁니다. 되돌릴 수 없습니다 | 결과를 내려받고 더 쓰지 않을 때 |

**끝내는 순서**

1. **결과 내려받기:** VS Code 탐색기에서 내려받을 폴더(`day65_딥러닝_학습원리`, `내작업`)를 오른쪽 클릭해 **Download...** 를 누르고 내 PC에 저장할 위치를 고릅니다. 폴더째 받아집니다.
2. **정지:** 파드 상세 칸의 **Stop** 을 누르고 확인 창의 **Stop Pod** 를 누릅니다. 볼륨이 없으면 창에 `ALL DATA will be lost!`(모든 데이터가 지워진다)가 뜨고, 정지 중 요금(Idle disk cost)은 0입니다.
3. **종료:** 정지되면 버튼이 **Start**, **Terminate**, **Request support** 로 바뀝니다. **Terminate** 를 누르고 확인 창의 **Terminate Pod** 를 누릅니다. 실행 중인 파드에는 Terminate 버튼이 없어서 먼저 정지해야 합니다.
4. Pods 화면에 파드가 남아 있지 않은지 확인합니다.

<img src="images/vscode_17_download.png" width="740" alt="VS Code 탐색기에서 단원 폴더를 오른쪽 클릭한 메뉴. 폴더와 Download... 항목에 빨간 사각형이 그려져 있다.">

<img src="images/runpod_08_stop.png" width="560" alt="정지 확인 창. 볼륨이 없어 모든 데이터가 지워진다는 경고와 Stop Pod 버튼에 빨간 사각형이 그려져 있다.">

<img src="images/runpod_09_terminate.png" width="600" alt="정지된 파드의 상세 칸. 정지한 뒤 나타난 Terminate 버튼에 빨간 사각형이 그려져 있다.">

- **다음에 파드를 새로 만들면:** 주소와 포트가 바뀌므로 5절(접속 명령 복사, 호스트 추가)부터 다시 합니다. 3절(키, 공개키 등록, Remote - SSH 확장)은 다시 하지 않습니다. 파드의 파일이 모두 지워졌으므로 5-4(파드에 확장 설치)와 6절도 다시 합니다.
- **지출 확인:** 콘솔의 **Billing** 에서 날짜별·파드별 사용 금액을 봅니다. 네트워크 볼륨은 파드를 지워도 남아 매달 요금이 나가므로, **Storage** 메뉴에서 안 쓰는 볼륨이 없는지 함께 봅니다.

## 9. VS Code로 접속이 안 될 때: Jupyter Lab

SSH 설정이 막혀 수업이 늦어질 것 같으면 브라우저의 Jupyter Lab으로 먼저 실습합니다. 파드는 그대로 씁니다.

1. 파드 상세 칸 **Connect** 탭의 **HTTP services** 에서 **Port 8888 → Jupyter Lab** 이 **Ready** 가 되면 링크를 누릅니다.
2. Jupyter Lab의 **File → New → Terminal** 에서 6-1(자료 받기)과 6-3(글꼴)을 그대로 실행합니다.
3. Jupyter Lab의 기본 커널 **Python 3 (ipykernel)** 에는 PyTorch가 이미 들어 있습니다. 나머지 두 라이브러리만 설치합니다.

**pip 명령(Jupyter Lab 터미널, 파이썬 패키지 설치 프로그램)**

```bash
# 템플릿 파이썬에 pandas 와 matplotlib 을 설치합니다
pip install pandas matplotlib
```

4. 왼쪽 파일 목록에서 교안을 열고 커널 **Python 3 (ipykernel)** 로 실행합니다. 결과는 파일을 오른쪽 클릭해 **Download** 로 받습니다.

<img src="images/runpod_05b_jupyter.png" width="598" alt="파드 상세 칸의 Connect 탭. HTTP services 의 Jupyter Lab 링크와 Ready 표시에 빨간 사각형이 그려져 있다.">

<img src="images/runpod_07_lesson_cuda.png" width="760" alt="Jupyter Lab에서 교안 01 준비 셀을 실행한 화면. 출력 계산 장치: cuda 와 GPU 이름(NVIDIA A40)에 빨간 사각형이 그려져 있다.">

## 10. 자주 막히는 곳

**접속**

- **VS Code가 비밀번호를 묻는다:** 공개키가 등록되지 않은 것입니다. 3-2에서 공개키 한 줄 전체(`ssh-ed25519 ...`)를 붙였는지 봅니다(`SHA256:` 로 시작하는 지문을 붙이면 안 됩니다). 파드를 만든 뒤에 등록했다면 파드를 종료하고 새로 만듭니다.
- **Connection timed out 이나 Connection refused:** 파드가 아직 시작 중일 수 있으니 1분 뒤 다시 접속합니다. Connect 탭에 **SSH over exposed TCP** 칸이 없으면 4-3의 SSH 설정이 꺼진 파드이므로 새로 만듭니다.
- **REMOTE HOST IDENTIFICATION HAS CHANGED:** 예전에 쓴 주소·포트가 새 파드에 다시 배정된 것입니다. 내 PC 터미널에서 옛 기록을 지우고 다시 접속합니다.

  **OpenSSH 명령(내 PC 터미널)**

  ```bash
  # [주소]:포트 를 경고에 나온 값으로 바꿉니다
  ssh-keygen -R "[38.147.83.27]:23887"
  ```

**환경과 커널**

- **커널 목록에 `.venv` 가 없다:** `/workspace` 를 연 상태입니다. 6-4대로 저장소 폴더를 엽니다. 그래도 없으면 저장소 폴더에서 `uv sync` 를 실행했는지 봅니다.
- **`No module named 'pandas'`(또는 `'torch'`):** 커널이 `.venv` 가 아닌 다른 파이썬입니다. 7-1대로 커널을 바꿉니다.
- **`uv sync` 가 디스크 부족으로 멈춘다:** 컨테이너 디스크가 작게 만들어진 파드입니다. 기본값 50GB로 새로 만듭니다.

**git·uv가 없다는 오류(`command not found`)**

RunPod PyTorch 템플릿에는 git·curl·uv가 들어 있습니다. 다른 템플릿을 써서 `git: command not found` 나 `uv: command not found` 가 나오면 먼저 설치하고 6-1부터 다시 합니다. 파이썬은 따로 설치하지 않습니다. `uv sync` 가 `.python-version` 에 적힌 Python 3.12를 찾고, 없으면 스스로 받아 옵니다.

**apt 명령(파드 터미널)**

```bash
# git 과 curl(인터넷에서 파일을 받는 프로그램)을 설치합니다
apt-get update && apt-get install -y git curl
```

**리눅스 셸 명령(파드 터미널)**

```bash
# curl 로 uv 공식 설치 스크립트를 받아 sh 로 실행합니다
curl -LsSf https://astral.sh/uv/install.sh | sh
```

```bash
# 지금 열린 터미널에서 uv 명령을 바로 쓸 수 있게 경로를 불러옵니다
source $HOME/.local/bin/env
```

**GPU와 비용**

- **`CUDA out of memory`:** 모델이나 배치가 GPU 메모리에 다 들어가지 않습니다. 배치 크기를 줄이거나 메모리가 더 큰 GPU로 파드를 새로 만듭니다.
- **배포 버튼이 Deploy when available로 바뀜:** 고른 GPU의 자리가 없다는 뜻입니다(카드에 Out of capacity). **Available** 탭에서 다른 GPU를 고릅니다. 버튼이 아예 눌리지 않으면 크레딧이 부족한지 봅니다.
- **정지했다 다시 켜니 GPU가 0개:** 원래 서버의 GPU를 다른 사람이 쓰고 있으면 GPU가 0개로 뜰 수 있습니다. 파드를 종료하고 새로 만듭니다.
- **그래프의 한글이 네모로 나옴:** 6-3의 글꼴을 설치한 뒤 노트북 위쪽 **Restart** 로 커널을 다시 시작합니다. 그래도 네모면 파드 터미널에서 리눅스 셸 명령 `rm -rf ~/.cache/matplotlib` 로 글꼴 목록을 지우고 다시 시작합니다.

## 11. 핵심 정리

- **처음 한 번:** SSH 키를 만들어 공개키를 RunPod에 등록하고, VS Code에 Remote - SSH 확장을 설치합니다.
- **파드마다:** SSH over exposed TCP 명령으로 호스트를 추가해 접속하고, 파드에 Python·Jupyter 확장을 설치합니다.
- **자료와 환경:** `git clone` 으로 저장소를 받고, 저장소 폴더에서 `uv sync` 로 `.venv` 를 만든 뒤 커널로 `.venv` 를 고릅니다.
- **요금:** 파드는 켜져 있는 동안 요금이 나갑니다. 결과를 내려받은 뒤 정지하고 종료합니다.